# Year 2012

In [1]:
import cocopp
dsl = cocopp.load("bbob/2012/*")

In [2]:
import numpy as np

dd = dsl.dictByDimFunc()     # your grouped datasets
t = 1e-8                     # choose the target precision

best_by_df = {}              # (dim, fid) -> (best_alg, best_ert)

for dim in sorted(dd.keys()): 
    for fid in sorted(dd[dim].keys()):
        rows = []
        for ds in dd[dim][fid]:                 # each ds = one algorithm
            ert = float(ds.detERT([t])[0])      # ERT in #evals at target t
            rows.append((ds.algId, ert))  
        # ignore INF (not reached) when picking best
        finite = [(a, e) for (a, e) in rows if np.isfinite(e)] 
    
        if finite:
            best_alg, best_ert = min(finite, key=lambda x: x[1]) 
        else:
            best_alg, best_ert = None, np.inf
        best_by_df[(dim, fid)] = (best_alg, best_ert) 
        print(f"dim={dim:>2}, F{fid:>2} -> {best_alg}  (ERT={best_ert:.3g} @ {t})")


dim= 2, F 1 -> DE-BFGS_voglis  (ERT=56.5 @ 1e-08)
dim= 2, F 2 -> PSO-BFGS_voglis  (ERT=95.4 @ 1e-08)
dim= 2, F 3 -> DE-ROLL_voglis  (ERT=714 @ 1e-08)
dim= 2, F 4 -> JADE_posik  (ERT=1.11e+03 @ 1e-08)
dim= 2, F 5 -> CMAmh_brockhoff  (ERT=15.5 @ 1e-08)
dim= 2, F 6 -> DE-AUTO_voglis  (ERT=199 @ 1e-08)
dim= 2, F 7 -> DE-SIMPLEX_voglis  (ERT=333 @ 1e-08)
dim= 2, F 8 -> DE-BFGS_voglis  (ERT=168 @ 1e-08)
dim= 2, F 9 -> PSO-BFGS_voglis  (ERT=149 @ 1e-08)
dim= 2, F10 -> DE-SIMPLEX_voglis  (ERT=156 @ 1e-08)
dim= 2, F11 -> DE-BFGS_voglis  (ERT=155 @ 1e-08)
dim= 2, F12 -> DE-BFGS_voglis  (ERT=210 @ 1e-08)
dim= 2, F13 -> DE-SIMPLEX_voglis  (ERT=182 @ 1e-08)
dim= 2, F14 -> DE-SIMPLEX_voglis  (ERT=153 @ 1e-08)
dim= 2, F15 -> DE-BFGS_voglis  (ERT=512 @ 1e-08)
dim= 2, F16 -> DE-SIMPLEX_voglis  (ERT=952 @ 1e-08)
dim= 2, F17 -> CMAma_brockhoff  (ERT=1.43e+03 @ 1e-08)
dim= 2, F18 -> JADE_posik  (ERT=2.08e+03 @ 1e-08)
dim= 2, F19 -> CMAmh_brockhoff  (ERT=2.27e+03 @ 1e-08)
dim= 2, F20 -> JADEb_posik  (ERT=1

In [3]:
from collections import Counter, defaultdict

In [4]:
# Build a frequency counter: how many (dim,fid) each algo wins
win_counter = Counter(
    alg for (alg, ert) in best_by_df.values()
    if alg is not None and np.isfinite(ert)
)

# If you want a plain dict:
wins_dict = dict(win_counter)

# (Optional) pretty print, most wins first
for alg, count in win_counter.most_common():
    print(f"{alg}: {count}")

DE-BFGS_voglis: 16
BIPOPsaACM_loshchilov: 16
IPOPsaACM_loshchilov: 14
JADE_posik: 13
NIPOPaCMA_loshchilov: 13
CMAma_brockhoff: 12
DE-AUTO_voglis: 10
PSO-BFGS_voglis: 9
DE-SIMPLEX_voglis: 8
DEAE_posik: 6
CMAmah_brockhoff: 6
CMAmh_brockhoff: 4
BIPOPaCMA_loshchilov: 2
DEctpb_posik: 2
CMAES_posik: 2
NBIPOPaCMA_loshchilov: 2
DE-ROLL_voglis: 1
JADEb_posik: 1
DBRCGA_chuang: 1
CMA_brockhoff: 1
DE_posik: 1
CMAa_brockhoff: 1
xNESas_schaul: 1
MVDE_melo: 1


In [5]:
"""
Given best_by_df: {(dim, fid): (alg, ert)},
return {dim: algo_with_most_(fid)_wins_in_that_dim}.
Tie-break: lower total ERT across that dim, then alphabetical.
    """
wins = defaultdict(Counter)                    # dim -> Counter({alg: count})
ert_sums = defaultdict(lambda: defaultdict(float))  # dim -> {alg: total_ert}

for (dim, fid), (alg, ert) in best_by_df.items():
    if alg is None or not np.isfinite(ert):
        continue
    wins[dim][alg] += 1
    ert_sums[dim][alg] += float(ert)

result = {}
for dim, counter in wins.items():
    max_wins = max(counter.values())
    candidates = [a for a, c in counter.items() if c == max_wins]
    best = min(candidates, key=lambda a: (ert_sums[dim][a], a))  # tie-breaks
    result[dim] = best
result


{2: 'DE-SIMPLEX_voglis',
 3: 'DE-BFGS_voglis',
 5: 'IPOPsaACM_loshchilov',
 10: 'IPOPsaACM_loshchilov',
 20: 'BIPOPsaACM_loshchilov',
 40: 'NIPOPaCMA_loshchilov'}

In [6]:
import numpy as np
import pandas as pd

# Make sure 'dd' already exists
# (if not, run: dsl = cocopp.load('path/to/your/ppdata'); dd = dsl.dictByDimFunc())

targets = [1e-1, 1e-2, 1e-3, 1e-5, 1e-8]
rows = []  # reset before starting the full loop

for dim in sorted(dd.keys()):                      # e.g. [2, 3, 5, 10, 20, 40]
    for fid in sorted(dd[dim].keys()):
        for t in targets:
            algo_erts = []
            for ds in dd[dim][fid]:                # each algorithm
                ert = float(ds.detERT([t])[0])
                algo_erts.append((ds.algId, ert))
            
            finite = [(a, e) for (a, e) in algo_erts if np.isfinite(e)]

            if finite:
                best_alg, best_ert = min(finite, key=lambda x: x[1])
            else:
                best_alg, best_ert = None, np.inf

            rows.append({
                "dimension": dim,
                "function_id": fid,
                "target": t,
                "best_algorithm": best_alg,
                "best_ERT": best_ert
            })

# Build DataFrame
df_best = pd.DataFrame(rows)
df_best = df_best.sort_values(by=["dimension", "function_id", "target"]).reset_index(drop=True)

# Confirm dimensions included
print("✅ Unique dimensions in table:", df_best["dimension"].unique())
print(df_best.head(15))


✅ Unique dimensions in table: [ 2  3  5 10 20 40]
    dimension  function_id        target    best_algorithm    best_ERT
0           2            1  1.000000e-08    DE-BFGS_voglis   56.533333
1           2            1  1.000000e-05    DE-BFGS_voglis   56.533333
2           2            1  1.000000e-03    DE-BFGS_voglis   56.533333
3           2            1  1.000000e-02  CMAmah_brockhoff   55.466667
4           2            1  1.000000e-01  CMAmah_brockhoff   34.333333
5           2            2  1.000000e-08   PSO-BFGS_voglis   95.400000
6           2            2  1.000000e-05   PSO-BFGS_voglis   87.333333
7           2            2  1.000000e-03   PSO-BFGS_voglis   81.800000
8           2            2  1.000000e-02    DE-AUTO_voglis   78.733333
9           2            2  1.000000e-01    DE-BFGS_voglis   74.066667
10          2            3  1.000000e-08    DE-ROLL_voglis  714.200000
11          2            3  1.000000e-05        JADE_posik  685.333333
12          2            3 

In [7]:
import os
os.makedirs("results", exist_ok=True)

df_best.to_csv("results/best_algos_2012.csv", index=False)
